In [ ]:
"""
================================================================================
  HYBRID MODEL: HistGradientBoosting + XGBoost — Stacking Ensemble
  Author   : ML Pipeline (merged from two base scripts)
  Strategy : Level-0 → HGB + XGB  |  Level-1 → Ridge meta-learner
================================================================================

WHY STACKING (not simple averaging)?
─────────────────────────────────────
• Blending/averaging assumes equal model contribution — rarely optimal.
• Stacking trains a meta-learner on out-of-fold predictions, letting it
  discover how much to trust each base model for different feature regions.
• HGB handles missing values natively and is strong on mid-range patterns.
• XGBoost is powerful on non-linear interactions and tail distributions.
• A Ridge meta-learner combines both without overfitting.

STRENGTHS:
  - Better generalisation than either model alone.
  - Meta-learner corrects systematic biases of base models.
  - Early stopping in XGBoost prevents overfitting.

WEAKNESSES / TRADE-OFFS:
  - Higher training time (OOF folds × 2 models).
  - More complexity → harder to interpret directly.
  - If base models are highly correlated, gains are modest.
"""

# ============================================================
# 0. IMPORTS
# ============================================================
import os
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")          # non-interactive backend for file saving
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, KFold
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import (
    mean_absolute_error, r2_score, explained_variance_score,
    median_absolute_error, max_error, mean_squared_log_error,
    confusion_matrix, classification_report, accuracy_score
)
import xgboost as xgb
import joblib

warnings.filterwarnings("ignore")
os.makedirs("models",  exist_ok=True)
os.makedirs("outputs", exist_ok=True)

SEED = 42
np.random.seed(SEED)

# ============================================================
# 1. LOAD DATA
# ============================================================
print("=" * 60)
print("  STEP 1 — LOADING DATA")
print("=" * 60)

df = pd.read_csv("data/cleaned/final_cleaned_dataset.csv")
print(f"  Shape: {df.shape}")
print(f"  Columns: {list(df.columns)}")

# ── Target detection ────────────────────────────────────────
target_candidates = [c for c in df.columns if "yield" in c.lower()]
if not target_candidates:
    raise ValueError("No 'yield' column found.")
target = target_candidates[0]
print(f"\n  Target column: '{target}'")

# ── Features ────────────────────────────────────────────────
exclude_cols = ["Country", "Year"]
features = sorted(df.columns.difference(exclude_cols + [target]).tolist())
X = df[features].copy()
y = df[target].copy()
print(f"  Features ({len(features)}): {features}")

# ============================================================
# 2. TRAIN / VALIDATION / TEST SPLIT  (60 / 20 / 20)
# ============================================================
print("\n" + "=" * 60)
print("  STEP 2 — SPLITTING DATA  (60 / 20 / 20, time-ordered)")
print("=" * 60)

X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.20, shuffle=False
)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.25, shuffle=False   # 0.25 × 0.80 = 0.20
)
print(f"  Train : {X_train.shape[0]} rows")
print(f"  Val   : {X_val.shape[0]} rows")
print(f"  Test  : {X_test.shape[0]} rows")

# ============================================================
# 3. ENCODE CATEGORICALS + SCALING
# ============================================================
print("\n" + "=" * 60)
print("  STEP 3 — ENCODING CATEGORICALS & SCALING (StandardScaler)")
print("=" * 60)

from sklearn.preprocessing import LabelEncoder

# Label-encode any object/category columns that slipped into features
X_train = X_train.copy()
X_val   = X_val.copy()
X_test  = X_test.copy()

cat_encoders = {}
for col in X_train.select_dtypes(include=["object", "category"]).columns:
    le = LabelEncoder()
    X_train[col] = le.fit_transform(X_train[col].astype(str))
    # unseen labels in val/test → map to 0 (or last class)
    def safe_transform(enc, vals):
        mapping = {v: i for i, v in enumerate(enc.classes_)}
        return np.array([mapping.get(str(v), 0) for v in vals])
    X_val[col]  = safe_transform(le, X_val[col])
    X_test[col] = safe_transform(le, X_test[col])
    cat_encoders[col] = le
    print(f"  Label-encoded '{col}' ({len(le.classes_)} classes)")

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_val_sc   = scaler.transform(X_val)
X_test_sc  = scaler.transform(X_test)
joblib.dump(scaler, "models/scaler_hybrid.pkl")
joblib.dump(cat_encoders, "models/cat_encoders.pkl")
print("  Scaler saved → models/scaler_hybrid.pkl")

# Combine train+val for final model training (after stacking OOF)
X_trainval_sc = np.vstack([X_train_sc, X_val_sc])
y_trainval    = pd.concat([y_train, y_val]).reset_index(drop=True)

# ============================================================
# 4. BASE MODEL A — HistGradientBoostingRegressor
# ============================================================
print("\n" + "=" * 60)
print("  STEP 4a — TRAINING HistGradientBoostingRegressor")
print("=" * 60)

t0 = time.time()
hgb = HistGradientBoostingRegressor(
    max_iter=500,
    max_depth=6,
    learning_rate=0.05,
    min_samples_leaf=5,
    l2_regularization=0.1,
    early_stopping=True,
    validation_fraction=0.15,
    n_iter_no_change=25,
    random_state=SEED,
    verbose=0
)
hgb.fit(X_train_sc, y_train)
hgb_train_time = time.time() - t0
hgb_iters = hgb.n_iter_
print(f"  HGB  → iterations: {hgb_iters} | train time: {hgb_train_time:.2f}s")
joblib.dump(hgb, "models/hgb_model.pkl")

# ============================================================
# 5. BASE MODEL B — XGBoost with Early Stopping
# ============================================================
print("\n" + "=" * 60)
print("  STEP 4b — TRAINING XGBoost with Early Stopping")
print("=" * 60)

xgb_params = {
    "objective":        "reg:squarederror",
    "eval_metric":      "rmse",
    "max_depth":        6,
    "eta":              0.05,
    "subsample":        0.8,
    "colsample_bytree": 0.8,
    "min_child_weight": 5,
    "reg_alpha":        0.1,
    "reg_lambda":       1.0,
    "seed":             SEED,
    "verbosity":        0,
}

dtrain = xgb.DMatrix(X_train_sc, label=y_train, feature_names=features)
dval   = xgb.DMatrix(X_val_sc,   label=y_val,   feature_names=features)
dtest  = xgb.DMatrix(X_test_sc,  label=y_test,  feature_names=features)

evals_result = {}
t0 = time.time()
bst = xgb.train(
    xgb_params,
    dtrain,
    num_boost_round=1000,
    evals=[(dtrain, "train"), (dval, "val")],
    early_stopping_rounds=30,
    evals_result=evals_result,
    verbose_eval=False
)
xgb_train_time = time.time() - t0
xgb_iters = bst.best_iteration
print(f"  XGB  → best iteration: {xgb_iters} | train time: {xgb_train_time:.2f}s")
bst.save_model("models/xgb_model.json")

# ============================================================
# 6. STACKING — OUT-OF-FOLD META-FEATURES
# ============================================================
print("\n" + "=" * 60)
print("  STEP 5 — GENERATING OOF PREDICTIONS (5-Fold Stacking)")
print("=" * 60)
"""
  Strategy:
  - Split train+val into 5 folds.
  - For each fold, train HGB + XGB on the other 4 folds, predict on
    the held-out fold. This gives unbiased OOF predictions.
  - Stack OOF predictions → [oof_hgb, oof_xgb] as meta-features.
  - Train a Ridge meta-learner on these meta-features.
  - Final test meta-features = average of 5 fold predictions on test set.
"""

kf = KFold(n_splits=5, shuffle=True, random_state=SEED)

oof_hgb = np.zeros(len(X_trainval_sc))
oof_xgb = np.zeros(len(X_trainval_sc))
test_preds_hgb = np.zeros((5, X_test_sc.shape[0]))
test_preds_xgb = np.zeros((5, X_test_sc.shape[0]))

y_tv = y_trainval.values

for fold, (tr_idx, va_idx) in enumerate(kf.split(X_trainval_sc)):
    print(f"  Fold {fold+1}/5 ...", end=" ")

    Xf_tr, Xf_va = X_trainval_sc[tr_idx], X_trainval_sc[va_idx]
    yf_tr, yf_va = y_tv[tr_idx], y_tv[va_idx]

    # ── HGB fold ────────────────────────────────────────────
    hgb_f = HistGradientBoostingRegressor(
        max_iter=500, max_depth=6, learning_rate=0.05,
        min_samples_leaf=5, l2_regularization=0.1,
        early_stopping=True, validation_fraction=0.15,
        n_iter_no_change=25, random_state=SEED, verbose=0
    )
    hgb_f.fit(Xf_tr, yf_tr)
    oof_hgb[va_idx]    = hgb_f.predict(Xf_va)
    test_preds_hgb[fold] = hgb_f.predict(X_test_sc)

    # ── XGB fold ────────────────────────────────────────────
    d_tr = xgb.DMatrix(Xf_tr, label=yf_tr, feature_names=features)
    d_va = xgb.DMatrix(Xf_va, label=yf_va, feature_names=features)
    bst_f = xgb.train(
        xgb_params, d_tr, num_boost_round=1000,
        evals=[(d_tr, "train"), (d_va, "val")],
        early_stopping_rounds=30, verbose_eval=False
    )
    oof_xgb[va_idx]    = bst_f.predict(xgb.DMatrix(Xf_va, feature_names=features))
    test_preds_xgb[fold] = bst_f.predict(xgb.DMatrix(X_test_sc, feature_names=features))

    print(f"HGB iters={hgb_f.n_iter_} | XGB best={bst_f.best_iteration}")

# Meta-features for training (OOF) and test (mean of folds)
meta_train = np.column_stack([oof_hgb, oof_xgb])
meta_test  = np.column_stack([test_preds_hgb.mean(axis=0),
                               test_preds_xgb.mean(axis=0)])

# ============================================================
# 7. META-LEARNER — Ridge Regression
# ============================================================ ///  essayer deux/3 autres algo   ///
print("\n" + "=" * 60)
print("  STEP 6 — TRAINING RIDGE META-LEARNER")
print("=" * 60)

ridge = Ridge(alpha=1.0)
ridge.fit(meta_train, y_tv)
print(f"  Ridge coefficients: HGB={ridge.coef_[0]:.4f}, XGB={ridge.coef_[1]:.4f}")
print(f"  (These indicate relative trust in each base model)")
joblib.dump(ridge, "models/ridge_meta.pkl")

# ============================================================
# 8. FINAL PREDICTIONS
# ============================================================
print("\n" + "=" * 60)
print("  STEP 7 — FINAL PREDICTIONS ON ALL SETS")
print("=" * 60)

# Train-set: use OOF meta-features (unbiased)
y_pred_stack_train = ridge.predict(meta_train)

# Test: use averaged fold predictions
y_pred_stack_test = ridge.predict(meta_test)

# Individual model predictions on test (for comparison)
y_pred_hgb_test = hgb.predict(X_test_sc)
y_pred_xgb_test = bst.predict(dtest)

print("  Stacking predictions generated.")

# ============================================================
# 9. METRICS
# ============================================================
def compute_metrics(y_true, y_pred, name="Dataset"):
    """Return a dict of all regression metrics and print them."""
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    rmse   = np.sqrt(np.mean((y_true - y_pred) ** 2))
    mae    = mean_absolute_error(y_true, y_pred)
    medae  = median_absolute_error(y_true, y_pred)
    r2     = r2_score(y_true, y_pred)
    ev     = explained_variance_score(y_true, y_pred)
    maxerr = max_error(y_true, y_pred)
    mape   = np.mean(np.abs((y_true - y_pred) / (y_true + 1e-8))) * 100
    smape  = (100 / len(y_true)) * np.sum(
                 2 * np.abs(y_pred - y_true) / (np.abs(y_pred) + np.abs(y_true) + 1e-8))
    try:
        msle = mean_squared_log_error(
            np.clip(y_true, 0, None), np.clip(y_pred, 0, None))
    except Exception:
        msle = np.nan

    print(f"\n{'─'*50}")
    print(f"  {name} METRICS")
    print(f"{'─'*50}")
    print(f"  RMSE              : {rmse:.4f}")
    print(f"  MAE               : {mae:.4f}")
    print(f"  Median Abs Error  : {medae:.4f}")
    print(f"  R² Score          : {r2:.4f}")
    print(f"  Explained Variance: {ev:.4f}")
    print(f"  Max Error         : {maxerr:.4f}")
    print(f"  MAPE (%)          : {mape:.2f}")
    print(f"  sMAPE (%)         : {smape:.2f}")
    print(f"  MSLE              : {msle:.6f}" if not np.isnan(msle) else "  MSLE              : N/A")

    return dict(rmse=rmse, mae=mae, medae=medae, r2=r2, ev=ev,
                maxerr=maxerr, mape=mape, smape=smape, msle=msle)

print("\n" + "=" * 60)
print("  STEP 8 — EVALUATION METRICS")
print("=" * 60)

metrics_train = compute_metrics(y_tv, y_pred_stack_train, "STACK — TRAIN (OOF)")
metrics_test  = compute_metrics(y_test, y_pred_stack_test,  "STACK — TEST")

print("\n── Baseline comparison on TEST ──")
m_hgb = compute_metrics(y_test, y_pred_hgb_test, "HGB alone — TEST")
m_xgb = compute_metrics(y_test, y_pred_xgb_test, "XGB alone — TEST")

# Training time summary
print(f"\n{'─'*50}")
print(f"  TRAINING TIME SUMMARY")
print(f"{'─'*50}")
print(f"  HGB  base model : {hgb_train_time:.2f}s  | iterations: {hgb_iters}")
print(f"  XGB  base model : {xgb_train_time:.2f}s  | best round: {xgb_iters}")

# ============================================================
# 10. REGRESSION → CLASSIFICATION  (for Confusion Matrix)
# ============================================================
"""
  BINNING STRATEGY
  ────────────────
  We divide the target (yield) into 4 quantile-based bins so each bin
  has roughly equal support:
      0 → Low yield    (Q0–Q25)
      1 → Medium yield (Q25–Q50)
      2 → High yield   (Q50–Q75)
      3 → Very high    (Q75–Q100)
  Quantile binning is preferred over equal-width because crop yield
  distributions are often skewed — quantile bins avoid empty classes.
"""
print("\n" + "=" * 60)
print("  STEP 9 — REGRESSION → CLASSIFICATION (4 quantile bins)")
print("=" * 60)

bin_edges = np.quantile(y_tv, [0, 0.25, 0.50, 0.75, 1.0])
bin_edges[0]  -= 1e-6
bin_edges[-1] += 1e-6
bin_labels = ["Low", "Medium", "High", "VeryHigh"]

def to_bins(values):
    return np.digitize(values, bin_edges[1:-1])   # → 0,1,2,3

y_test_cls  = to_bins(np.array(y_test))
y_pred_cls  = to_bins(y_pred_stack_test)

cm = confusion_matrix(y_test_cls, y_pred_cls)
acc = accuracy_score(y_test_cls, y_pred_cls)

print(f"\n  Bin edges: {np.round(bin_edges, 2)}")
print(f"  Classification Accuracy : {acc:.4f}")
print(f"\n  Classification Report:\n")
print(classification_report(y_test_cls, y_pred_cls,
                             target_names=bin_labels, zero_division=0))

# ============================================================
# 11. VISUALIZATIONS
# ============================================================
print("\n" + "=" * 60)
print("  STEP 10 — GENERATING VISUALIZATIONS")
print("=" * 60)

PALETTE = ["#2E86AB", "#E84855", "#3BB273", "#F4A261"]
plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor":   "white",
    "axes.grid":        True,
    "grid.alpha":       0.3,
    "font.size":        10,
    "axes.titlesize":   12,
    "axes.titleweight": "bold",
})

# ── Fig 1: Actual vs Predicted (all 3 models) ───────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("Actual vs Predicted — TEST SET", fontsize=14, fontweight="bold", y=1.02)

for ax, y_p, title, color in zip(
        axes,
        [y_pred_hgb_test, y_pred_xgb_test, y_pred_stack_test],
        ["HGB", "XGBoost", "Hybrid Stack"],
        PALETTE[:3]):
    y_t = np.array(y_test)
    ax.scatter(y_t, y_p, alpha=0.45, s=20, color=color, edgecolors="none")
    lims = [min(y_t.min(), y_p.min()), max(y_t.max(), y_p.max())]
    ax.plot(lims, lims, "k--", lw=1.5, label="Perfect fit")
    r2v = r2_score(y_t, y_p)
    ax.set_title(f"{title}  (R²={r2v:.4f})")
    ax.set_xlabel("Actual")
    ax.set_ylabel("Predicted")
    ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig("outputs/01_actual_vs_predicted.png", dpi=150, bbox_inches="tight")
plt.close()
print("  Saved → outputs/01_actual_vs_predicted.png")

# ── Fig 2: Residual Plots ────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("Residual Plots — TEST SET", fontsize=14, fontweight="bold")

for ax, y_p, title, color in zip(
        axes,
        [y_pred_hgb_test, y_pred_xgb_test, y_pred_stack_test],
        ["HGB", "XGBoost", "Hybrid Stack"],
        PALETTE[:3]):
    residuals = np.array(y_test) - y_p
    ax.scatter(y_p, residuals, alpha=0.45, s=18, color=color, edgecolors="none")
    ax.axhline(0, color="red", lw=1.5, linestyle="--")
    ax.set_title(f"{title} Residuals")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Residual")
    rmse_v = np.sqrt(np.mean(residuals**2))
    ax.text(0.05, 0.95, f"RMSE={rmse_v:.3f}", transform=ax.transAxes,
            va="top", fontsize=9, color="darkred")
plt.tight_layout()
plt.savefig("outputs/02_residual_plots.png", dpi=150, bbox_inches="tight")
plt.close()
print("  Saved → outputs/02_residual_plots.png")

# ── Fig 3: Error Distribution ────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("Error Distribution — TEST SET", fontsize=14, fontweight="bold")

for ax, y_p, title, color in zip(
        axes,
        [y_pred_hgb_test, y_pred_xgb_test, y_pred_stack_test],
        ["HGB", "XGBoost", "Hybrid Stack"],
        PALETTE[:3]):
    errors = np.array(y_test) - y_p
    ax.hist(errors, bins=40, color=color, alpha=0.75, edgecolor="white")
    ax.axvline(0, color="red", lw=1.5, linestyle="--")
    ax.axvline(errors.mean(), color="navy", lw=1.2, linestyle=":", label=f"Mean={errors.mean():.2f}")
    ax.set_title(f"{title}")
    ax.set_xlabel("Error (Actual − Predicted)")
    ax.set_ylabel("Count")
    ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig("outputs/03_error_distribution.png", dpi=150, bbox_inches="tight")
plt.close()
print("  Saved → outputs/03_error_distribution.png")

# ── Fig 4: Feature Importances ───────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(18, max(5, len(features) * 0.4 + 2)))
fig.suptitle("Feature Importances", fontsize=14, fontweight="bold")

# HGB — permutation importance fallback for sklearn versions without feature_importances_
try:
    hgb_imp = hgb.feature_importances_
except AttributeError:
    from sklearn.inspection import permutation_importance as _perm
    _pi = _perm(hgb, X_test_sc, np.array(y_test), n_repeats=8, random_state=SEED, n_jobs=-1)
    hgb_imp = _pi.importances_mean
order_hgb = np.argsort(hgb_imp)
axes[0].barh([features[i] for i in order_hgb], hgb_imp[order_hgb],
             color=PALETTE[0], alpha=0.85)
axes[0].set_title("HistGradientBoosting")
axes[0].set_xlabel("Importance")

# XGB
xgb_scores = bst.get_score(importance_type="gain")
xgb_imp = np.array([xgb_scores.get(f, 0) for f in features])
xgb_imp_norm = xgb_imp / (xgb_imp.sum() + 1e-10)
order_xgb = np.argsort(xgb_imp_norm)
axes[1].barh([features[i] for i in order_xgb], xgb_imp_norm[order_xgb],
             color=PALETTE[1], alpha=0.85)
axes[1].set_title("XGBoost (gain, normalised)")
axes[1].set_xlabel("Normalised Importance")

plt.tight_layout()
plt.savefig("outputs/04_feature_importances.png", dpi=150, bbox_inches="tight")
plt.close()
print("  Saved → outputs/04_feature_importances.png")

# ── Fig 5: Confusion Matrix Heatmap ─────────────────────────
fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=bin_labels, yticklabels=bin_labels,
            linewidths=0.5, linecolor="white",
            cbar_kws={"shrink": 0.8}, ax=ax)
ax.set_title(f"Confusion Matrix — Hybrid Stack\nAccuracy: {acc:.4f}", fontsize=13, fontweight="bold")
ax.set_xlabel("Predicted Class", fontsize=11)
ax.set_ylabel("True Class", fontsize=11)
plt.tight_layout()
plt.savefig("outputs/05_confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.close()
print("  Saved → outputs/05_confusion_matrix.png")

# ── Fig 6: XGBoost Learning Curve ────────────────────────────
train_rmse_curve = evals_result["train"]["rmse"]
val_rmse_curve   = evals_result["val"]["rmse"]
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(train_rmse_curve, label="Train RMSE", color=PALETTE[0], lw=1.5)
ax.plot(val_rmse_curve,   label="Val RMSE",   color=PALETTE[1], lw=1.5, linestyle="--")
ax.axvline(xgb_iters, color="green", lw=1.2, linestyle=":", label=f"Best round = {xgb_iters}")
ax.set_title("XGBoost Learning Curve (Early Stopping)", fontsize=13, fontweight="bold")
ax.set_xlabel("Boosting Round")
ax.set_ylabel("RMSE")
ax.legend()
plt.tight_layout()
plt.savefig("outputs/06_xgb_learning_curve.png", dpi=150, bbox_inches="tight")
plt.close()
print("  Saved → outputs/06_xgb_learning_curve.png")

# ── Fig 7: Metrics Comparison Bar Chart ──────────────────────
metric_names = ["RMSE", "MAE", "MAPE (%)", "R²"]
hgb_vals   = [m_hgb["rmse"], m_hgb["mae"], m_hgb["mape"], m_hgb["r2"]]
xgb_vals   = [m_xgb["rmse"], m_xgb["mae"], m_xgb["mape"], m_xgb["r2"]]
stack_vals = [metrics_test["rmse"], metrics_test["mae"], metrics_test["mape"], metrics_test["r2"]]

x = np.arange(len(metric_names))
width = 0.26

fig, ax = plt.subplots(figsize=(11, 5))
ax.bar(x - width, hgb_vals,   width, label="HGB",    color=PALETTE[0], alpha=0.85)
ax.bar(x,         xgb_vals,   width, label="XGB",    color=PALETTE[1], alpha=0.85)
ax.bar(x + width, stack_vals, width, label="Hybrid", color=PALETTE[2], alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(metric_names, fontsize=11)
ax.set_title("Model Comparison — TEST SET Metrics", fontsize=13, fontweight="bold")
ax.set_ylabel("Value")
ax.legend()
for bars in ax.containers:
    ax.bar_label(bars, fmt="%.3f", fontsize=7, padding=2)
plt.tight_layout()
plt.savefig("outputs/07_metrics_comparison.png", dpi=150, bbox_inches="tight")
plt.close()
print("  Saved → outputs/07_metrics_comparison.png")

# ── Fig 8: Per-Crop Performance (if available) ───────────────
if "Crop" in df.columns:
    crops_test = df.loc[X_test.index, "Crop"].values
    unique_crops = pd.Series(crops_test).unique()
    crop_r2, crop_rmse, crop_names = [], [], []
    for crop in unique_crops:
        mask = crops_test == crop
        if mask.sum() > 2:
            r2c   = r2_score(np.array(y_test)[mask], y_pred_stack_test[mask])
            rmse_c = np.sqrt(np.mean((np.array(y_test)[mask] - y_pred_stack_test[mask])**2))
            crop_r2.append(r2c)
            crop_rmse.append(rmse_c)
            crop_names.append(crop)

    if crop_names:
        order = np.argsort(crop_r2)
        fig, axes = plt.subplots(1, 2, figsize=(16, max(5, len(crop_names) * 0.5 + 2)))
        axes[0].barh([crop_names[i] for i in order], [crop_r2[i] for i in order],
                     color=PALETTE[0], alpha=0.85)
        axes[0].set_title("Per-Crop R² (Hybrid)")
        axes[0].set_xlabel("R²")
        axes[1].barh([crop_names[i] for i in order], [crop_rmse[i] for i in order],
                     color=PALETTE[1], alpha=0.85)
        axes[1].set_title("Per-Crop RMSE (Hybrid)")
        axes[1].set_xlabel("RMSE")
        plt.suptitle("Per-Crop Performance — TEST SET", fontsize=13, fontweight="bold")
        plt.tight_layout()
        plt.savefig("outputs/08_per_crop_performance.png", dpi=150, bbox_inches="tight")
        plt.close()
        print("  Saved → outputs/08_per_crop_performance.png")

        print("\n=== PER-CROP PERFORMANCE (Hybrid Stack) ===")
        for n, r, m in sorted(zip(crop_names, crop_r2, crop_rmse), key=lambda x: -x[1]):
            print(f"  {n:15}: R²={r:.4f}  RMSE={m:.2f}")

# ============================================================
# 12. FINAL SUMMARY
# ============================================================
print("\n" + "=" * 60)
print("  FINAL SUMMARY")
print("=" * 60)
print(f"""
  ┌──────────────────────────────────────────────┐
  │           TEST SET — MODEL COMPARISON        │
  ├────────────┬──────────┬──────────┬───────────┤
  │ Metric     │   HGB    │   XGB    │  Hybrid   │
  ├────────────┼──────────┼──────────┼───────────┤
  │ RMSE       │ {m_hgb['rmse']:8.4f} │ {m_xgb['rmse']:8.4f} │ {metrics_test['rmse']:9.4f} │
  │ MAE        │ {m_hgb['mae']:8.4f} │ {m_xgb['mae']:8.4f} │ {metrics_test['mae']:9.4f} │
  │ R²         │ {m_hgb['r2']:8.4f} │ {m_xgb['r2']:8.4f} │ {metrics_test['r2']:9.4f} │
  │ MAPE (%)   │ {m_hgb['mape']:8.2f} │ {m_xgb['mape']:8.2f} │ {metrics_test['mape']:9.2f} │
  │ sMAPE (%)  │ {m_hgb['smape']:8.2f} │ {m_xgb['smape']:8.2f} │ {metrics_test['smape']:9.2f} │
  ├────────────┴──────────┴──────────┴───────────┤
  │ Confusion-matrix accuracy (4 bins): {acc:.4f}   │
  └──────────────────────────────────────────────┘

  Models saved:
    models/hgb_model.pkl
    models/xgb_model.json
    models/ridge_meta.pkl
    models/scaler_hybrid.pkl

  Visualisations:
    outputs/01_actual_vs_predicted.png
    outputs/02_residual_plots.png
    outputs/03_error_distribution.png
    outputs/04_feature_importances.png
    outputs/05_confusion_matrix.png
    outputs/06_xgb_learning_curve.png
    outputs/07_metrics_comparison.png
    outputs/08_per_crop_performance.png  (if Crop column present)
""")

print("  ✓ Hybrid model pipeline complete.")

  STEP 1 — LOADING DATA
  Shape: (3289, 43)
  Columns: ['Year', 'Yield_kg_ha', 'Fert_P_tonnes', 'Fert_N_tonnes', 'Fert_Total_tonnes', 'Avg_Temperature_C', 'CO2_Emissions_TonsCapita', 'Sea_Level_Rise_mm', 'Rainfall_mm', 'Population', 'Renewable_Energy', 'Extreme_Weather_Events', 'Forest_Area', 'Yield_lag1', 'Yield_lag2', 'Yield_lag3', 'Yield_kg_ha_roll3_mean', 'Yield_kg_ha_roll3_std', 'Temp_lag1', 'Temp_anomaly', 'Rain_lag1', 'Rain_anomaly', 'Fert_per_capita', 'Year_norm', 'Year_norm2', 'crop_Maize', 'crop_Rice', 'crop_Wheat', 'ctry_Argentina', 'ctry_Australia', 'ctry_Brazil', 'ctry_Canada', 'ctry_China', 'ctry_France', 'ctry_Germany', 'ctry_India', 'ctry_Indonesia', 'ctry_Japan', 'ctry_Mexico', 'ctry_Russia', 'ctry_South Africa', 'ctry_UK', 'ctry_USA']

  Target column: 'Yield_kg_ha'
  Features (41): ['Avg_Temperature_C', 'CO2_Emissions_TonsCapita', 'Extreme_Weather_Events', 'Fert_N_tonnes', 'Fert_P_tonnes', 'Fert_Total_tonnes', 'Fert_per_capita', 'Forest_Area', 'Population', 'Rain_ano